# Kvantifiera delat designarv i en transformatorflotta med PROC INBREED

## Sammanfattning

En nätoperatörs stationstransformatorer konstrueras i successiva designgenerationer, där varje ny modell härleds från två föregående designer. Denna notebook behandlar den tekniska släkttavlan som en stamtavla och använder **PROC INBREED** för att beräkna inavels- och additiva släktskapskoefficienter (kovarians), vilket kvantifierar hur mycket designarv två tillgångar delar.

Stamtavlan är uppbyggd så att strukturen är tolkningsbar: två av de fyra nuvarande flottmodellerna härstammar från ett par **syskon**-föregångardesigner och bär därför på ett koncentrerat arv, medan de övriga bygger på disjunkta släktlinjer. Proceduren återskapar detta exakt. De två syskonhärledda modellerna (`G3_T1`, `G3_T3`) har vardera en inavelskoefficient på **F = 0,25**; de andra två (`G3_T2`, `G3_T4`) har **F = 0**. Flottans genomsnittliga inavelskoefficient är **0,0417**. I den additiva släktskapsmatrisen är det minst släktade nuvarande flottparet **`G3_T1` och `G3_T3` (släktskap = 0)** — de delar inget gemensamt ursprung och bildar den starkaste redundanta (N-1-)parningen, vilket är viktigt eftersom `G3_T1` själv är en av de mest inavlade designerna.

## Datakällor

| Datamängd | Rader | Nyckelvariabler | Beskrivning |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | En liten, deterministisk teknisk stamtavla för stationstransformatordesigner över tre icke överlappande designgenerationer (4 grundplattformar, 4 andragenerationens härledningar, 4 nuvarande flottmodeller). Varje icke-grundläggande design registrerar de två distinkta föregångardesigner (`Parent1`, `Parent2`) den härletts från. `Sex` markerar den ledande designrollen (M = mekanisk/kärnsläktled, F = elektrisk/lindningssläktled). Stamtavlan är handspecificerad — inte slumpmässig — så att släktskapsstrukturen är känd i förväg och kan kontrolleras mot procedurens utdata.

# Kvantifiera delat designarv i en transformatorflotta

## Varför ett elbolag bryr sig om "inavel"

En överförings- och distributionsoperatör driver hundratals stationstransformatorer. För att kontrollera ingenjörskostnad och kvalificeringsarbete konstruerar tillverkare sällan varje transformator från grunden — varje ny modell **ärver** kärngeometri, lindningstopologi, isolationssystem och genomföringsdesigner från en eller två föregående modeller. Över flera upphandlingscykler skapar detta en genuin *teknisk släkttavla*: en stamtavla över designer.

Detta delade arv är en dold tillförlitlighetsrisk. Om två transformatorer som skyddar samma last (ett redundant **N-1**-par) härstammar från samma anfäderdesign är det troligt att en latent designbrist — en resonansmod, en isolationsåldringsmekanism, en genomföringsöverslagsväg — finns i **båda** enheterna. En enda grundorsak kan då slå ut det redundanta paret samtidigt: ett *gemensamt-fel* (common-mode failure).

**PROC INBREED** byggdes för att kvantifiera precis den här typen av delat ursprung. Även om dess ursprung ligger i djur- och växtförädling är dess matematik — Wrights additiva släktskapsrekursion — domänoberoende: den mäter hur stor andel av designarvet två individer delar via gemensamma anfäder. Vi mappar genetikens vokabulär mot tillgångskonstruktion:

| INBREED-koncept | Teknisk tolkning |
|---|---|
| Individ | En transformatordesign / modell |
| Förälder (far/mor) | En föregångardesign den härletts från |
| Generation (CLASS) | En upphandlings-/designcykel |
| Inavelskoefficient *F* | Grad av självlikt arv inom en design |
| Kovarians-/släktskapskoefficient | Delat designarv mellan två designer |
| Minst släktade par | Bästa redundanta parning för N-1-motståndskraft |

## Steg 1 — Specificera designstamtavlan

Vi anger `DesignLineage` direkt så att släktskapsstrukturen är entydig. Det finns tre **icke överlappande designgenerationer**:

- **Generation 1** — fyra grundläggande plattformsdesigner (`G1_P1`-`G1_P4`) utan föregångare (tomma föräldrar).
- **Generation 2** — fyra härledda designer (`G2_D1`-`G2_D4`), var och en konstruerad från två **distinkta** Generation 1-plattformar. `G2_D1` och `G2_D2` är *helsyskon* (båda från `G1_P1` x `G1_P2`); `G2_D3` och `G2_D4` är helsyskon (båda från `G1_P3` x `G1_P4`).
- **Generation 3** — fyra nuvarande flottmodeller (`G3_T1`-`G3_T4`). `G3_T1` är byggd från syskonparet `G2_D1` x `G2_D2`, och `G3_T3` från syskonparet `G2_D3` x `G2_D4`; dessa två designer koncentrerar därför arvet. `G3_T2` och `G3_T4` korsar var och en de två disjunkta släktlinjerna.

Kolumnerna `ID`, `Parent1` och `Parent2` bär stamtavlan; `Sex` registrerar vilken teknisk släktlinje som ledde designen. Ett kort efterföljande DATA-steg tömmer platshållaren `.` så att grundplattformarna får tomma föräldrar, vilket PROC INBREED förväntar sig.

In [1]:
data DesignLineage;
   LÄNGD ID $12 Parent1 $12 Parent2 $12 Sex $1;
   INFILE DATALINES truncover;
   INDATA Generation ID $ Parent1 $ Parent2 $ Sex $;
   DATALINES;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
KÖR;

/* Grundplattformarna saknar föregångare: rensa platshållar-punkten */
data DesignLineage;
   STÄLL_IN DesignLineage;
   OM Parent1 = '.' SÅ Parent1 = ' ';
   OM Parent2 = '.' SÅ Parent2 = ' ';
KÖR;

TITEL 'Transformatordesignens stamtavla';
PROCEDUR SKRIV data=DesignLineage noobs ETIKETT;
   VARIABEL Generation ID Parent1 Parent2 Sex;
   ETIKETT Generation="Generation" ID="ID" Parent1="Förälder 1" Parent2="Förälder 2" Sex="Kön";
KÖR;


                                            Transformatordesignens stamtavla                                            


Generation     ID    Förälder 1    Förälder 2   Kön
----------  -----  ------------  ------------  ----
         1  G1_P1                              M
         1  G1_P2                              M
         1  G1_P3                              M
         1  G1_P4                              F
         2  G2_D1  G1_P1         G1_P2         M
         2  G2_D2  G1_P1         G1_P2         F
         2  G2_D3  G1_P3         G1_P4         F
         2  G2_D4  G1_P3         G1_P4         M
         3  G3_T1  G2_D1         G2_D2         M
         3  G3_T2  G2_D1         G2_D3         F
         3  G3_T3  G2_D3         G2_D4         F
         3  G3_T4  G2_D2         G2_D4         M




NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to Transformatordesignens stamtavla.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## Steg 2 — Inavelskoefficienter per designgeneration

Vi kör PROC INBREED i **flergenerationsläge** genom att ange `Generation` i `CLASS`-satsen, vilket skriver ut stamtavlesammanfattningen per generation. `VAR`-satsen anger de tre stamtavlekolumnerna i den ordning som krävs: individ, första föregångare, andra föregångare.

- **IND**-alternativet skriver ut varje designs inavelskoefficient *F* — ett mått på hur koncentrerat dess eget arv är. En design byggd från två nära besläktade föregångare bär en hög *F*.
- **MATRIX**-alternativet skriver ut hela den additiva släktskapsmatrisen så att vi kan läsa parvist arv direkt.
- **AVERAGE**-alternativet lägger till flottans genomsnittliga inavelskoefficient — en enda mångfaldssammanfattning för designen.

Värden nära 0 innebär oberoende släktlinjer; *F* stiger när en designs två föregångare blir mer nära besläktade.

In [2]:
TITEL 'Inavelskoefficienter per designgeneration';
PROCEDUR inbreed data=DesignLineage ind average MATRIX;
   VARIABEL ID Parent1 Parent2;
   KLASS Generation;
KÖR;


                                       Inavelskoefficienter per designgeneration                                        


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (Generation 2)
--------------------------------------
ID                  F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coe


NOTE: Option TITLE changed to Inavelskoefficienter per designgeneration.
NOTE: PROC INBREED data=DesignLineage



## Steg 3 — Kovarianskoefficienter sparade för nedströms riskbedömning

Genom att ersätta inavelsvyn med alternativet **COVAR** rapporteras samma relationer som **kovarians- (additiva släktskaps-) koefficienter**, den form ett team för tillgångsförvaltning skulle mata in i en redundans-riskpoäng. Alternativet **OUTCOV=** fångar hela matrisen som en datamängd (`DesignCovar`), där kolumnerna `Col1`-`Col12` innehåller varje designs släktskap till varje annan design (i stamtavleordning) — redo att kopplas tillbaka mot stationstilldelningar.

Vi skriver ut den sparade datamängden så att den sparade matrisen syns tillsammans med listan.

In [3]:
TITEL 'Kovarians- (släktskaps-) koefficienter';
PROCEDUR inbreed data=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   VARIABEL ID Parent1 Parent2;
   KLASS Generation;
KÖR;

TITEL 'Släktskapsmatris sparad via OUTCOV= för riskbedömning';
PROCEDUR SKRIV data=DesignCovar noobs;
KÖR;


                                         Kovarians- (släktskaps-) koefficienter                                         


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (Generation 1)
--------------------------------------------------
ID                     G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.00


NOTE: Option TITLE changed to Kovarians- (släktskaps-) koefficienter.
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to Släktskapsmatris sparad via OUTCOV= för riskbedömning.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## Steg 4 — Minst släktade parningar för redundanta (N-1-)installationer

Den sparade matrisen `DesignCovar` är exakt vad en redundansstudie behöver. Släktskapet mellan design *i* och design *j* finns i rad *i*, kolumn `Col`*j* (kolumnerna är i stamtavleordning, så `Col9`-`Col12` motsvarar de fyra nuvarande flottmodellerna `G3_T1`-`G3_T4`).

Vi läser ut de fyra nuvarande flottraderna från `DesignCovar`, bildar varje oordnat flottpar, kopplar på dess släktskapskoefficient och sorterar minst släktad först. Målet är att välja redundanta par vars delade arv är **lägst** — dessa minimerar risken att en ärvd designbrist slår ut båda halvorna av en N-1-skyddad position.

In [4]:
/* Hämta de fyra aktuella flottans design-rader; Col9..Col12 är släktskapet
   till G3_T1..G3_T4 (kolumnordning enligt stamtavlan). Bygg varje ordnat par. */
data Pairs;
   STÄLL_IN DesignCovar;
   DÄR ID IN ('G3_T1','G3_T2','G3_T3','G3_T4');
   LÄNGD DesignA $12 DesignB $12;
   FÄLT g3{4} Col9 Col10 Col11 Col12;
   FÄLT nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   DesignA = ID;
   GÖR j = 1 TILL 4;
      DesignB = nm{j};
      OM DesignA < DesignB SÅ GÖR;
         Relationship = g3{j};
         UTDATA;
      SLUT;
   SLUT;
   BEHÅLL DesignA DesignB Relationship;
KÖR;

PROCEDUR SORTERA data=Pairs; EFTER Relationship; KÖR;

TITEL 'Kandidatpar för redundans (N-1), minst släkt först';
PROCEDUR SKRIV data=Pairs noobs ETIKETT;
   VARIABEL DesignA DesignB Relationship;
   ETIKETT DesignA="Design A" DesignB="Design B" Relationship="Släktskapsgrad";
KÖR;
TITEL;


                                   Kandidatpar för redundans (N-1), minst släkt först                                   


Design A  Design B   Släktskapsgrad
--------  --------  ---------------
G3_T1     G3_T3                   0
G3_T2     G3_T4                0.25
G3_T1     G3_T2               0.375
G3_T1     G3_T4               0.375
G3_T2     G3_T3               0.375
G3_T3     G3_T4               0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to Kandidatpar för redundans (N-1), minst släkt först.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## Tolkning av resultaten

**Inavelskoefficienter (Steg 2).** Alla grundläggande Generation 1-plattformar och alla Generation 2-härledningar visar *F* = 0 — per konstruktion har ingen två besläktade föregångare. Två Generation 3-modeller uppvisar *F* = 0,25: `G3_T1` (byggd från syskonparet `G2_D1` x `G2_D2`) och `G3_T3` (från syskonparet `G2_D3` x `G2_D4`). Deras föregångare spårar tillbaka till *samma* par grundplattformar, så deras arv är koncentrerat. Ur tillförlitlighetssynpunkt är dessa de designer som är mest exponerade för en enskild ärvd defekt, och de motiverar extra oberoende kvalificeringstestning. `G3_T2` och `G3_T4`, som korsar de två disjunkta släktlinjerna, har *F* = 0.

**Släktskapsmatris (Steg 3).** Icke-diagonalposterna kvantifierar parvist delat arv. De två syskon-Generation 2-paren visar vardera ett släktskap på 0,50 sinsemellan (`G2_D1`-`G2_D2` och `G2_D3`-`G2_D4`), medan härledningar från motsatta släktlinjer ligger på 0,00. De inavlade nuvarande flottdesignerna bär självsläktskap på 1,25 (= 1 + *F*), synligt på diagonalen. Datamängden `DesignCovar` gör hela matrisen tillgänglig att koppla mot stationstilldelningar.

**Minst släktade parningar (Steg 4).** Att rangordna varje nuvarande flottpar efter släktskap placerar **`G3_T1` och `G3_T3` först med släktskap 0,00** — de härstammar från helt disjunkta anfädersplattformar och delar inget designarv. Detta är den starkaste redundanta parningen, och den är särskilt värdefull eftersom `G3_T1` själv är en av de två mest inavlade designerna: att para den med en helt obesläktad partner säkrar dess koncentrerade arv. Nästbästa par är `G3_T2` och `G3_T4` på 0,25; de återstående paren ligger på 0,375. Flottans genomsnittliga inavelskoefficient på **0,0417** (skriven ut av AVERAGE-alternativet i Steg 2) sammanfattar den övergripande designmångfalden. Upphandlingen bör föredra paren med lägst släktskap för N-1-positioner och, över tid, bredda den genetiska/anfädersbasen — motsvarigheten inom tillgångskonstruktion till att undvika en genetisk flaskhals.

**Reservation.** Detta är illustrativ syntetisk data; en produktionsstudie skulle bygga stamtavlan från tillverkarens verkliga designrevisionsregister och validera släktskapspoängen mot historiska gemensamma-fel-händelser innan den styr upphandlingsbeslut.